#대학 등록금 데이터 전처리 템플릿 (2010~2023)

In [ ]:
import pandas as pd
from pathlib import Path
# 파일 경로를 수동으로 지정
file_paths = [
    "2010년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2011년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2012년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2013년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2014년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2015년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2016년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2017년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2018년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2019년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2020년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2021년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2022년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2023년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx"
]
file_paths

In [ ]:

# 사이버대 제거 키워드
cyber_keywords = ['사이버', '방송통신']
dfs = {}

for path in file_paths:
    try:
        year = int(Path(path).stem[:4])
        df = pd.read_excel(path, header=3)
        df = df[~df['학교'].str.contains('|'.join(cyber_keywords), na=False)]
        df = df.rename(columns={'학교': '학교명'})
        df['기준년도'] = year
        dfs[year] = df
    except Exception as e:
        print(f"X {path} 처리 중 오류 발생:", e)

In [ ]:
processed = []
school_sets = [set(df['학교명']) for df in dfs.values()]
common_schools = set.intersection(*school_sets) if school_sets else set()

for year, df in dfs.items():
    df = df[df['학교명'].isin(common_schools)].copy()

    if '등록금\n(D=B+C)' in df.columns:
        df['등록금'] = df['등록금\n(D=B+C)']
    elif '등록금\n(D=B)' in df.columns:
        df['등록금'] = df['등록금\n(D=B)']
    else:
        df['등록금'] = 0

    if '입학금\n(A)' in df.columns:
        df['입학금\n(A)'] = pd.to_numeric(df['입학금\n(A)'], errors='coerce').fillna(0)
    else:
        df['입학금\n(A)'] = pd.Series(0, index=df.index)

    df['등록금'] = pd.to_numeric(df['등록금'], errors='coerce').fillna(0) + df['입학금\n(A)']
    df['등록금'] = df['등록금'].round(1) / 1000  # 소수점 첫째자리까지 천원단위 환산

    if '지역' not in df.columns:
        df['지역'] = '미상'
    if '설립구분' not in df.columns:
        df['설립구분'] = '미상'

    keep_cols = ['학교명', '기준년도', '등록금', '지역', '설립구분', '수업료\\(B)', '인문사회', '자연과학', '예체능', '공학', '의학']
    for col in keep_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[keep_cols]
    processed.append(df)

final_df = pd.concat(processed, ignore_index=True)
final_df.to_csv("최종_등록금_통합_2010_2023_from_xlsx.csv", index=False, encoding="utf-8-sig")
final_df.head()
